# reg/tile 选型

两种 Vector 侧计算方式的边界并不模糊，核心判断只有一个：**计算链的长度与 UB 往返的代价**。


## 1. 选型决策：其实是三档

```text
单步操作（add/mul/relu…）
        │
        ▼
   Tile 接口（pl.*）—— 一行一操作，接口即文档，中间结果写回 UB

固定两步组合（add+relu、mul+add…）
        │
        ▼
   官方融合 Tile 接口（pl.add_relu / pl.fused_mul_add / pl.axpy…）—— 接口级融合
   —— 框架预先封装好的"免手写 VF"中间态（03.03 扩展接口面）

多步计算链、经分析值得减少 UB 往返
        │
        ▼
   手写 VF 函数（vf.load → 链内计算 → vf.store）—— 函数级融合
   —— 尽量让中间结果留在寄存器；需考虑寄存器压力
```

核心判断仍然只有一个：**中间结果是否值得留在寄存器**——短链写回 UB 的代价可忽略；长链时"读写字节数 / 有效 UB 带宽"才显著。至于 Cube 与 Vector 协同的"算子级融合"，属于第 5 章及以后的内容，本章只区分前两档。

对照表（浓缩 03.02/03.03）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">判断维度</th><th style="text-align:left;">Tile 计算</th><th style="text-align:left;">Reg 计算</th></tr>
<tr><td>中间结果路径</td><td>每步经 UB</td><td>链内直通寄存器</td></tr>
<tr><td>开发成本</td><td>低（一行一操作）</td><td>中（VF 函数 + 掩码管理）</td></tr>
<tr><td>适用算子</td><td>elementwise、简单归约</td><td>多段激活链、elementwise+归约融合段</td></tr>
<tr><td>数据路径</td><td>GM→UB→GM</td><td>GM→UB→RF→UB→GM</td></tr>
</table>


## 2. 本质关系：同一硬件的两层封装

一个常见的疑问：Tile 接口和 VF 是不是一回事？答案是**分层的两层，不是上下调用**：

- **硬件层**：两者最终都跑在 AIV 的 Vector 单元（寄存器）上——"本质都是 vector 计算"成立；
- **框架 IR 层**：它们是两套**平行的算子族**（tile 接口一族、`vf.*` 一族），各有独立的编译产码路径。Tile 接口（`pl.add` 等）在编译时被展开成完整的"UB 取数 → 寄存器算 → 写回 UB"序列——你感知不到寄存器，代价是未融合的中间结果通常落 UB；
- **VF 的意义**：把"寄存器级的序列"开放给你手写——load 一次、链内算完、store 一次，中间结果不落 UB。

所以三档的关系是：**Tile 接口封装块级计算；官方融合接口封装特定组合；VF 开放寄存器级序列**——同一硬件，三种控制粒度。这解释了为什么选型表里 Reg 计算能省 UB 往返：省的不是计算本身，而是未融合计算链中的中间读写；实际访存和收益仍需查看产物与性能数据。


## 3. 一个具体例子

`y = relu(x + b) * 0.5 + x`（残差缩放，4 个操作）：

- 按 4 个独立 Tile 调用估算：add 读 x/b，relu 与 mul 各读一次中间值，最后 add 读中间值/x，合计约 6 次整块读取、4 次整块写入；
- VF 按寄存器片段处理：各加载一次 x/b，保留 x 供残差使用，最终写一次结果，理想情况下为 2 次读取、1 次写入。

这是接口级流量估算，融合 Tile API、寄存器溢出和实际产码会改变结果，不能直接当作耗时比。

操作数越多，Reg 版的 UB 带宽节省越显著；但单操作或两操作的短链，Reg 版反而增加 VF 函数与掩码的书写成本——**短链用 Tile、长链用 Reg**。


## 4. 两条补充判断

1. **混用是常态**：一个 Kernel 中，简单段用 Tile 接口、热点段用 VF 函数，按段选择而不是全盘二选一；
2. **先用 Tile 跑通、再对热点上 Reg**：功能正确性优先（第 8 章性能分析的流程会再次强调这个顺序）。


### 比较前先固定数学语义

对本章 [8,256] FP32，单次整块读写都是 8192 B。VF 版本若每行只加载四个 x 寄存器、只写四个 y 寄存器，其算法级 UB 输入输出流量为 16 KiB；Tile 版本还包含平方中间量、归约临时区以及统计量访问。不能用“接口数量比例”直接作为加速比：寄存器溢出、归约指令、同步与数据搬运都会影响耗时。

还要保持 RMSNorm 的 eps 和除 N 一致。把 `relu(x)*alpha+beta` 改成 `relu(alpha*x+beta)` 会改变负数附近的结果，并非合法性能优化。比较时先在随机值、全零、负常量和小幅值上通过同一 Golden，再测量预热后的执行时间。


## 5. 课后练习

1. （选择）"每一层都写一个 VF 函数"是否正确做法？
   A. 是，Reg 计算总是更快<br/>
   B. 否，应先用 Tile 实现保证正确，性能分析后再把热点段改写为 VF
2. （实践）分析 RMSNorm 的计算链（均方根归约 + 归一化），判断：哪几步适合 Tile 接口？如果上 Reg，寄存器里要保留什么中间量？

In [ ]:
!cat ./answer/03.04_answer.txt

## 6. 本节小结

- 选型一句话：**单步用 Tile、固定组合用官方融合接口、长链手写 VF，按段混用**（本质是同一硬件的三种控制粒度）；
- Reg 的潜在收益来自减少 UB 读写字节数，最终耗时须实测；
- 工程顺序：Tile 跑通 → 性能分析 → 热点改 VF。

下一节：用本章全部知识实现 RMSNorm。